In [ ]:
!rm -rf /content/drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
ZIP = '/content/drive/MyDrive/tensors_rgb_256_packed_k.zip'
!ls -lh "$ZIP"
!mkdir -p /content/data
!unzip -q "$ZIP" -d /content/data
!ls /content/data

-rw------- 1 root root 1.5G Sep 15 11:30 /content/drive/MyDrive/tensors_rgb_256_packed_k.zip
tensors_rgb_256_packed


In [ ]:
from google.colab import files
uploaded = files.upload()
!ls *.py

Saving config.py to config.py
Saving dataset.py to dataset.py
Saving evaluate.py to evaluate.py
config.py  dataset.py  evaluate.py


In [ ]:
from google.colab import files
uploaded = files.upload()
!ls *.py

Saving targets.py to targets.py
Saving train.py to train.py
Saving yolo_stride16.py to yolo_stride16.py
config.py  dataset.py  evaluate.py  targets.py	train.py  yolo_stride16.py


In [ ]:
!apt-get install -qq python3.11 python3.11-venv python3.11-dev > /dev/null 2>&1
!python3.11 -m venv /content/akv
!/content/akv/bin/pip install -q --upgrade pip
!/content/akv/bin/pip install -q akida-models==1.14.2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 15.7 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
DATA = '/content/data/tensors_rgb_256_packed'
import re

src = open('config.py').read()
src = re.sub(r"^ROOT = Path\(.*?\)$", "ROOT = Path('/content')", src, flags=re.M)
src = re.sub(r"^TENSOR_DIR = .*$", f"TENSOR_DIR = Path('{DATA}')", src, flags=re.M)
src = re.sub(r"^SPLITS_FILE = .*$", f"SPLITS_FILE = Path('{DATA}/splits.json')", src, flags=re.M)
src = re.sub(r"^RUNS_DIR = .*$", "RUNS_DIR = Path('/content/runs')", src, flags=re.M)
open('config.py','w').write(src)

src = open('dataset.py').read()
src = src.replace(
    'if not npy.exists() or not meta_path.exists():',
    'npz = tensor_dir / f"{clip}_tensors.npz"\n    if not meta_path.exists() or (not npy.exists() and not npz.exists()):'
)
src = src.replace(
    'tensors = np.load(npy, mmap_mode="r")',
    'tensors = np.load(npy, mmap_mode="r") if npy.exists() else np.load(npz)["a"]'
)
open('dataset.py','w').write(src)

!grep -n "ROOT\|TENSOR_DIR\|SPLITS_FILE\|RUNS_DIR\|INPUT_SIZE\|^GRID" config.py

18:ROOT = Path('/content')
19:# TENSOR_DIR = ROOT / "Data_new" / "tensors_rgb"
20:# TENSOR_DIR = ROOT / "Data_new" / "tensors_messy"
21:# TENSOR_DIR = ROOT / "Data_new" / "tensors_rgb_448"
22:TENSOR_DIR = Path('/content/data/tensors_rgb_256_packed')
23:SPLITS_FILE = Path('/content/data/tensors_rgb_256_packed/splits.json')
24:RUNS_DIR = Path('/content/runs')
31:#INPUT_SIZE = 224
32:INPUT_SIZE = 256
33:#INPUT_SIZE = 448
38:GRID = 16
39:CELL = INPUT_SIZE / GRID  # 32 px either way
44:SCALE = min(INPUT_SIZE / SRC_W, INPUT_SIZE / SRC_H)     # 0.70 at 448
45:PAD_X = (INPUT_SIZE - SRC_W * SCALE) / 2                # 0.0
46:PAD_Y = (INPUT_SIZE - SRC_H * SCALE) / 2                # 44.8


In [ ]:
!MPLBACKEND=Agg /content/akv/bin/python -u dataset.py

tensors : /content/data/tensors_rgb_256_packed
policy  : keep

TRAIN  (90 clips in split)
  clips loaded : 90
  samples      : 28045
  quiet frames : 0 (policy: keep)
  boxes        : 28345
  box size     : median 11.8 px (0.74 cells), p5 7.7, p95 20.2
  under 8 px   : 6.9%

VALIDATION  (12 clips in split)
  clips loaded : 12
  samples      : 3728
  quiet frames : 0 (policy: keep)
  boxes        : 3728
  box size     : median 11.7 px (0.73 cells), p5 7.4, p95 20.8
  under 8 px   : 10.6%

TEST  (12 clips in split)
  clips loaded : 12
  samples      : 3723
  quiet frames : 0 (policy: keep)
  boxes        : 4243
  box size     : median 12.1 px (0.76 cells), p5 9.2, p95 23.4
  under 8 px   : 1.0%

One batch:
  images shape : (4, 256, 256, 3)  float32
  value range  : 0.00 to 232.00
  zero pixels  : 19.9%
  boxes/frame  : [1, 1, 1, 1]
  out of bounds: 0
  inside padding: 0


In [ ]:
import subprocess, threading, time, os
os.makedirs('/content/drive/MyDrive/drone_runs', exist_ok=True)

def backup():
    while True:
        time.sleep(600)
        subprocess.run('cp -r /content/runs/* /content/drive/MyDrive/drone_runs/ 2>/dev/null', shell=True)

threading.Thread(target=backup, daemon=True).start()

!MPLBACKEND=Agg /content/akv/bin/python -u train.py --epochs 15 --lr 1e-3 --batch_size 64 --name full_rgb_s16_256

2026-09-15 13:04:28.021429: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1789477468.248708    6184 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1789477468.314176    6184 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1789477468.775046    6184 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1789477468.775103    6184 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1789477468.775110    6184 computation_placer.cc:177] computation placer alr

In [ ]:
!MPLBACKEND=Agg /content/akv/bin/python -u evaluate.py --run full_rgb_s16_256 --split validation

2026-09-15 14:17:56.740519: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1789481876.769144   25350 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1789481876.777835   25350 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1789481876.803364   25350 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1789481876.803408   25350 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1789481876.803412   25350 computation_placer.cc:177] computation placer alr

In [ ]:
!MPLBACKEND=Agg /content/akv/bin/python -u evaluate.py --run full_rgb_s16_256 --split test
!cp -r /content/runs/* /content/drive/MyDrive/drone_runs/
!cd /content && zip -qr full_rgb_s16_256.zip runs/full_rgb_s16_256
!cp /content/full_rgb_s16_256.zip /content/drive/MyDrive/
!ls -lh /content/full_rgb_s16_256.zip

2026-09-15 14:18:47.031820: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1789481927.052583   26304 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1789481927.059675   26304 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1789481927.075939   26304 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1789481927.075984   26304 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1789481927.075988   26304 computation_placer.cc:177] computation placer alr

In [ ]:
from google.colab import files
uploaded = files.upload()
!ls *.py

Saving diagnose_confidence.py to diagnose_confidence (1).py
 config.py   'diagnose_confidence (1).py'   evaluate.py   train.py
 dataset.py   diagnose_confidence.py	    targets.py	  yolo_stride16.py


In [ ]:
!MPLBACKEND=Agg /content/akv/bin/python -u diagnose_confidence.py --run full_rgb_s16_256 --split validation

2026-09-15 14:42:38.934721: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1789483358.957644   33010 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1789483358.964695   33010 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1789483358.984301   33010 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1789483358.984328   33010 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1789483358.984333   33010 computation_placer.cc:177] computation placer alr

In [ ]:
!mv "diagnose_confidence (1).py" diagnose_confidence.py
!grep -n "ev.nms\|ev.decode" diagnose_confidence.py

150:            dets = ev.decode_predictions(p, conf_threshold=args.conf)
151:            dets = ev.nms(dets, threshold=args.nms)


In [ ]:
!MPLBACKEND=Agg /content/akv/bin/python -u diagnose_confidence.py --run full_rgb_s16_256 --split validation

2026-09-15 14:45:11.468005: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1789483511.491756   33712 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1789483511.504128   33712 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1789483511.530056   33712 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1789483511.530100   33712 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1789483511.530106   33712 computation_placer.cc:177] computation placer alr

In [ ]:
!MPLBACKEND=Agg /content/akv/bin/python -u evaluate.py --run full_rgb_s16_256 --split validation --conf 0.85

2026-09-15 14:46:21.289550: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1789483581.311471   34053 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1789483581.318737   34053 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1789483581.336506   34053 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1789483581.336555   34053 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1789483581.336559   34053 computation_placer.cc:177] computation placer alr

In [ ]:
!sed -n '159,212p' evaluate.py

def average_precision(matches, n_truth):
    """
    Area under the precision-recall curve, computed over all confidence
    thresholds rather than one arbitrary cut.

    A single-threshold number can be tuned to look good; AP cannot.
    """
    if n_truth == 0:
        return 0.0, [], []

    matches = sorted(matches, key=lambda m: -m[0])

    tp = 0
    fp = 0

    precisions = []
    recalls = []

    for score, is_tp in matches:
        if is_tp:
            tp += 1
        else:
            fp += 1

        precisions.append(tp / (tp + fp))
        recalls.append(tp / n_truth)

    if not precisions:
        return 0.0, [], []

    # Interpolate: precision at each recall is the best achievable at
    # that recall or higher. Standard for AP and stops noise in the
    # curve depressing the number.
    for i in range(len(precisions) - 2, -1, -1):
        precisions[i] = max(precisions[i], precisions[i + 1])

    ap = 0.0
    prev_r = 0.0

    for p, r in zip(precisions, recalls):